# Image sensors: CCD and CMOS {#sec-image}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/image-sensors.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/image-sensors.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter5/detlib.py', 'detlib.py')
else:
    sys.path.insert(0, '..')                 # detlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import detlib as dl
dl.use_style()

*Slides 108–111 (CCD and CMOS image sensors, image sensing with a pixel array, colour imaging with a Bayer filter or three CCDs),
112 (active-matrix readout), 113 (CMOS pixel, microlenses and colour filters), 114–117 (the CCD element and three-phase charge
transfer) and 118–120 (full-frame, frame-transfer and interline-transfer architectures).*

## Key relations

An image sensor is an array of photosensitive pixels, each integrating the photogenerated charge during the exposure time
(slide 110). The signal of a pixel, in electrons, is

$$
S=\eta\,\Phi_{ph}\,A_\text{pix}\,t_\text{exp}\le N_\text{FW},
$$

with $\eta$ the QE (including the fill factor and the microlens), $\Phi_{ph}$ the photon flux density, and $N_\text{FW}$ the
**full-well capacity**. The noise combines the photon shot noise of the signal, the shot noise of the dark charge $D$ and the read
noise $\sigma_r$ of the output amplifier (in electrons rms):

$$
\text{SNR}=\frac{S}{\sqrt{S+D+\sigma_r^2}},\qquad \text{dynamic range}=\frac{N_\text{FW}}{\sigma_r}.
$$

**CCD.** Each pixel is a MOS capacitor: a positive gate voltage on p-type Si creates a depletion region, a potential well that
stores the photogenerated electrons (slide 115). By clocking three interleaved gate phases the wells move and the packets are
shifted, without mixing, one gate at a time (slides 116–117): three clock steps move a packet by one pixel. The packets are shifted
row by row into a serial register and then pixel by pixel to a single, very low-noise output amplifier (slide 118). A packet that
is transferred $N$ times with a **charge-transfer efficiency** CTE keeps a fraction $\text{CTE}^N$. Frame-transfer and interline
architectures (slides 119–120) add a light-shielded storage area so the next exposure can start while the previous frame is read.

**CMOS.** Each pixel has its own photodiode and transistors (reset, source follower, row select: the 3T pixel of slide 113) and is
addressed like a memory (row and column decoders, active matrix of slide 112). There is no charge transfer across the chip, the
readout is fast and parallel, and the logic, ADCs and image processing are integrated on the same chip; the price was historically
a higher read noise and fixed-pattern noise, now largely overcome. **Colour** is obtained with a Bayer mosaic of red, green and blue
filters (twice as many green as red or blue pixels, like the eye's sensitivity) or, in professional cameras, with three sensors
behind a trichroic prism (slide 111).

## Live panel: three-phase charge transfer (slides 116–117)

```{ojs}
//| echo: false
viewof ccd = Inputs.form({
  step: Inputs.range([0, 18], {value: 0, step: 1, label: "clock step (t₁, t₂, …)"}),
  cte: Inputs.range([0.9, 1], {value: 0.99, step: 0.001, label: "charge-transfer efficiency per gate transfer"})
})
```

```{ojs}
//| echo: false
ccdOut = {
  const phases = [[0], [0, 1], [1], [1, 2], [2], [2, 0]], s = ccd.step, act = phases[s % 6], nG = 15;
  const gates = d3.range(nG).map(g => ({g, ph: g % 3, on: act.includes(g % 3)}));
  const shift = [0, 0.5, 1, 1.5, 2, 2.5][s % 6] + 3 * Math.floor(s / 6), Q0 = [10, 6, 14];
  const packets = Q0.map((q, k) => ({x: 3 * k + shift, q: q * ccd.cte ** Math.floor((s + 1) / 2), lost: q * (1 - ccd.cte ** Math.floor((s + 1) / 2))}));
  const table = ["V₁", "V₂", "V₃"].map((v, p) => `${v} = ${act.includes(p) ? "+V" : "0"}`).join(" · ");
  return {gates, packets, table};
}
Plot.plot({
  width: Math.min(width, 1050), height: 230, x: {label: "gate position", domain: [-0.5, 14.5]}, y: {label: "electron potential energy", domain: [-1.4, 0.4], ticks: []},
  marks: [Plot.rect(ccdOut.gates, {x1: d => d.g - 0.45, x2: d => d.g + 0.45, y1: 0.25, y2: 0.35, fill: d => d.on ? "#e34948" : "#c3c2b7"}),
          Plot.text(ccdOut.gates, {x: "g", y: 0.38, text: d => `G${d.g + 1}`, fontSize: 9}),
          Plot.rect(ccdOut.gates, {x1: d => d.g - 0.5, x2: d => d.g + 0.5, y1: d => d.on ? -1 : -0.05, y2: 0, fill: "#2a78d6", fillOpacity: 0.15}),
          Plot.dot(ccdOut.packets.filter(p => p.x < 14.6), {x: "x", y: -0.85, r: d => 3 + 1.5 * Math.sqrt(d.q), fill: "#2a78d6"}),
          Plot.text(ccdOut.packets.filter(p => p.x < 14.6), {x: "x", y: -1.2, text: d => `${d.q.toFixed(2)} (lost ${d.lost.toFixed(2)})`, fontSize: 9})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">step t<sub>${ccd.step + 1}</sub>: ${ccdOut.table} · packets moved by ${(([0, 0.5, 1, 1.5, 2, 2.5][ccd.step % 6] + 3 * Math.floor(ccd.step / 6)) / 3).toFixed(2)} pixel</div>`
```

Each packet spreads under two gates when both are high (t₂, t₄, …) and is squeezed under the next one when the previous gate
drops. After six steps every packet has moved by three gates, i.e. one pixel: the packet that started under G1 is under G4. The
packets never mix because a low gate always separates them. The lost charge does not vanish: it is left behind and appears in the
trailing pixels as a smear.

## Live panel: a simulated exposure

The scene below is converted into photoelectrons with Poisson statistics, the read noise is added, the full well clips bright pixels,
and the CCD transfer loss smears charge along the readout direction (to the right and downward in this sketch).

```{ojs}
//| echo: false
import {rng} from "./det_sim.js"
viewof cam = Inputs.form({
  lph: Inputs.range([0, 5.5], {value: 2.5, step: 0.05, label: "log₁₀ photons per pixel at the brightest point"}),
  qe: Inputs.range([0.1, 1], {value: 0.6, step: 0.01, label: "quantum efficiency (incl. fill factor)"}),
  rn: Inputs.range([0, 50], {value: 5, step: 0.5, label: "read noise σ_r (e⁻ rms)"}),
  fw: Inputs.range([500, 100000], {value: 20000, step: 500, label: "full-well capacity (e⁻)"}),
  lcte: Inputs.range([2, 7], {value: 6, step: 0.1, label: "−log₁₀(1 − CTE)"}),
  seed: Inputs.range([1, 50], {value: 1, step: 1, label: "noise seed"})
})
```

```{ojs}
//| echo: false
camOut = {
  const N = 160, r = rng(cam.seed), cte = 1 - 10 ** (-cam.lcte), phMax = 10 ** cam.lph, scene = new Float64Array(N * N);
  for (let y = 0; y < N; y++) for (let x = 0; x < N; x++) {
    let v = 0.05 + 0.25 * x / N;                                          // background gradient
    if (((x / 8) | 0) % 2 === 0 && y > 10 && y < 50) v = 0.3 + 0.7 * (x / N);   // bar target
    const dx = x - 110, dy = y - 105; if (dx * dx + dy * dy < 900) v = 1.0;     // bright disc
    if (Math.abs(x - 40) < 16 && Math.abs(y - 110) < 16) v = 0.02 * (1 + (x - 24) / 4);   // dark ramp patch
    scene[y * N + x] = v;
  }
  const e = new Float64Array(N * N);
  for (let i = 0; i < N * N; i++) e[i] = Math.min(r.poisson(cam.qe * phMax * scene[i]) + cam.rn * r.gauss(), cam.fw);
  // CCD transfer loss: pixel (x, y) undergoes y parallel + x serial transfers, deferred charge goes to the next pixel read
  const out = new Float64Array(N * N);
  for (let y = 0; y < N; y++) { let carry = 0; for (let x = 0; x < N; x++) {
    const i = y * N + x, keep = cte ** (3 * (x + y + 2)); out[i] = e[i] * keep + carry; carry = e[i] * (1 - keep); } }
  const vmax = Math.max(1, Math.min(cam.fw, cam.qe * phMax * 1.05));
  const cvs = DOM.canvas(N, N), ctx = cvs.getContext("2d"), img = ctx.createImageData(N, N);
  for (let i = 0; i < N * N; i++) { const g = Math.max(0, Math.min(255, 255 * Math.sqrt(Math.max(out[i], 0) / vmax))); img.data[4 * i] = img.data[4 * i + 1] = img.data[4 * i + 2] = g; img.data[4 * i + 3] = 255; }
  ctx.putImageData(img, 0, 0); cvs.style.width = "420px"; cvs.style.imageRendering = "pixelated"; cvs.style.border = "1px solid #c3c2b7";
  const patch = []; for (let y = 55; y < 70; y++) for (let x = 120; x < 150; x++) patch.push(out[y * N + x]);
  const S = d3.mean(patch), sd = d3.deviation(patch);
  return {cvs, S, sd, cte, theory: cam.qe * phMax * (0.05 + 0.25 * 135 / N)};
}
camOut.cvs
```

```{ojs}
//| echo: false
html`<div class="result-box">flat background patch: mean ${camOut.S.toFixed(1)} e⁻, noise ${camOut.sd.toFixed(1)} e⁻ rms → <b>SNR = ${(camOut.S / camOut.sd).toFixed(1)}</b>
(prediction S/√(S + σ<sub>r</sub>²) = ${(camOut.theory / Math.sqrt(camOut.theory + cam.rn ** 2)).toFixed(1)}) · dynamic range N<sub>FW</sub>/σ<sub>r</sub> = ${cam.rn > 0 ? (20 * Math.log10(cam.fw / cam.rn)).toFixed(0) + " dB" : "∞"} ·
CTE = ${camOut.cte.toFixed(7)}, a corner pixel keeps ${(100 * camOut.cte ** (3 * 320)).toFixed(1)} % of its charge after ${3 * 320} gate transfers</div>`
```

Things to try: at 10 photons per pixel the image is dominated by Poisson noise and the bars are barely visible; at 10⁴ it is clean.
Set the read noise to 30 e⁻: the dark patch disappears in the noise at low light, while bright parts are unaffected (read noise
matters only in the shadows). Reduce the full well: the disc saturates and its edges flatten (no "blooming" is modelled). Reduce
the CTE to 0.999: the bright disc smears to the right and the far corner loses most of its charge; space-qualified CCDs degrade
this way under radiation damage.

## Worked example: a professional full-frame CCD (slide 108)

> The Teledyne-DALSA FTF6040C has an imaging area of 36 × 24 mm² with 6 µm × 6 µm pixels. How many pixels? How many transfers does
> the farthest pixel undergo (three gates per pixel)? What CTE is needed to keep 99 % of its charge? How long does a readout take at
> 40 Mpixel/s through one output?

In [ ]:
#| label: tbl-ccd
#| tbl-cap: "Numbers behind the FTF6040C CCD of slide 108."
nx, ny = round(36e-3/6e-6), round(24e-3/6e-6)
Ntr = 3*(nx + ny)
cte_needed = 0.99**(1/Ntr)
rows = [("pixels", f"{nx} × {ny} = {nx*ny/1e6:.0f} Mpixel"), ("gate transfers of the farthest pixel", f"{Ntr}"),
        ("CTE for 99 % retention", f"{cte_needed:.7f} (1 − CTE = {1 - cte_needed:.1e})"),
        ("retention with CTE = 0.99999", f"{dl.cte_remaining(0.99999, Ntr):.1%}"),
        ("retention with CTE = 0.9999", f"{dl.cte_remaining(0.9999, Ntr):.1%}"),
        ("readout time at 40 Mpixel/s (one output)", f"{nx*ny/40e6:.2f} s")]
pd.DataFrame(rows, columns=["quantity", "value"])

A CCD needs a CTE of 0.9999997 or better: 30 electrons lost per 100 million transferred. This astonishing figure is why CCDs were
made in dedicated processes, and why CMOS sensors, which do not move charge across the chip, took over consumer imaging.

## Photon transfer curve

In [ ]:
#| label: fig-ptc
#| fig-cap: "Photon transfer curve of a simulated sensor (QE 0.6, read noise 5 e⁻, full well 20 000 e⁻, PRNU 1 %): noise versus signal. Three regimes: read-noise floor (slope 0), photon shot noise (slope 1/2), fixed-pattern noise (slope 1); the curve collapses at full well. Engineers use it to measure the gain (e⁻/DN), read noise and full well of a camera."
rng = np.random.default_rng(1)
S = np.logspace(-1, 4.6, 40)
noise = []
for s in S:
    prnu = 1 + 0.01*rng.standard_normal(20000)
    e = np.minimum(rng.poisson(s*prnu) + rng.normal(0, 5, 20000), 20000)
    noise.append((e.mean(), e.std()))
noise = np.array(noise)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.loglog(noise[:, 0].clip(0.1), noise[:, 1], "o", ms=4, label="simulated")
s = np.logspace(-1, 4.3, 100)
ax.loglog(s, np.full_like(s, 5.0), ":", label="read noise 5 e⁻"); ax.loglog(s, np.sqrt(s), ":", label="shot noise √S")
ax.loglog(s, 0.01*s, ":", label="PRNU 1 % × S")
ax.set(xlabel="mean signal (e⁻)", ylabel="noise (e⁻ rms)", ylim=(1, 500)); ax.legend(fontsize=8); plt.show()

::: {.callout-tip collapse="true"}
## Going further
- **Pixel size and diffraction.** A lens of f-number $N$ blurs a point into an Airy disc of diameter $2.44\lambda N$ (Ch. 1). For 1 µm
  smartphone pixels at f/1.8 and 550 nm, how many pixels does the Airy disc cover? Compare with the Nyquist condition (two pixels per
  resolved period). Are 200 Mpixel phone cameras resolution-limited by the pixels or by the optics?
- **Bayer demosaicing and aliasing.** With one colour per pixel, each colour is under-sampled; fine patterns produce colour moiré.
  Simulate a Bayer mosaic of a chirped grating and interpolate it bilinearly.
- **Backside illumination, deep trench isolation, stacked sensors.** Modern CMOS sensors are thinned and lit from the back so that the
  wiring does not shadow the photodiode (QE > 80 %), separated by trenches against optical and electrical crosstalk, and bonded onto
  a second chip that carries the logic.
- **Single-photon imagers.** Electron-multiplying CCDs (an avalanche register before the amplifier, @sec-apd-noise) and SPAD arrays
  (@sec-apd) reach sub-electron read noise. What excess noise factor does an EMCCD have, and why does it effectively halve the QE
  at high light levels?
- **Rolling shutter.** CMOS sensors read row by row; fast motion is sheared. Estimate the skew of a propeller filmed with a 30 ms
  readout.
:::

::: {.callout-note collapse="true"}
## Complements
- Kasap, sec. 5.13; J. R. Janesick, *Photon Transfer* (SPIE Press, 2007): the bible of sensor characterisation.
- E. R. Fossum, "CMOS image sensors: electronic camera-on-a-chip", *IEEE Trans. Electron Devices* 44, 1689 (1997).
- W. S. Boyle and G. E. Smith, *Bell Syst. Tech. J.* 49, 587 (1970) (slide 2); Nobel Prize in Physics 2009.
:::

## Try it yourself

1. A smartphone pixel of 1 µm² with QE 0.7 is exposed for 1/60 s to an indoor scene giving 10⁻⁴ W/m² at 550 nm on the sensor. How
   many electrons does it collect? What SNR can be achieved with 2 e⁻ read noise?
2. Why are there twice as many green pixels as red or blue in a Bayer filter? (Hint: photometry, Ch. 3.)
3. With the exposure panel, find the photon level at which read noise and shot noise contribute equally for $\sigma_r=5$ e⁻.
   Express it in terms of $\sigma_r$.
4. An interline-transfer CCD (slide 120) has a fill factor of only 30 %. How do microlenses (slide 113) restore the QE, and why do
   they fail for light arriving at large angles (fast lenses, image corners)?